# 第7周 · 周六：AB实验综合实战——端到端实验

> 难度：⭐⭐⭐⭐⭐ | 定位：整合本周全部知识，完成从设计到决策的完整AB实验
> 前置依赖：W7D1-D5全部内容 | 后续用途：真实工作中的实验设计和分析
> 面试地位：现场编程题的经典场景——"这是数据，分析这个实验的结果"
> 配对课程：01python_learn W7D6（相关分析与热力图）

---

## 📘 区块1：核心概念——综合实战的目标

### 1.1 今天要做什么

我们要完成两个完整的AB实验：

**实验1：新用户注册引导流程**
- 场景：产品想测试"3步渐进式引导"vs"直接进入首页"
- 主要指标：7日留存率
- 涉及知识：假设检验、样本量计算、SRM检测、效应量分析

**实验2：推荐算法迭代**
- 场景：算法团队升级了推荐模型
- 主要指标：人均GMV（连续指标）
- 涉及知识：连续指标的假设检验、CUPED简化版、异质性分析

### 1.2 综合实战的检查清单

一个完整的AB实验分析应该覆盖以下所有检查项：

```
实验前 ✓
  □ 假设是否清晰（三要素完整）
  □ 样本量是否事前计算（不是事后找补）
  □ 主要指标是否事前定义（只有一个）
  □ 决策规则是否事前约定

实验中 ✓
  □ 是否固定样本量，不Peeking
  □ 是否监控了数据质量和护栏指标
  □ 是否记录了实验日志

实验后 ✓
  □ SRM检验是否通过
  □ 护栏指标是否全部通过
  □ 报告了p值+效应量+置信区间
  □ 没有事后修改主要指标
  □ 子群分析标注为探索性
```

> **今天最重要的认知**：一个完整的AB实验不只是"p<0.05就上线"。要检查SRM（分流对不对）、护栏指标（有没有副作用）、效应量的业务意义（提升够不够大）、决策规则是否事前约定。实验结论是一个"多维度判断"，不是"一个p值判断"。

### 1.3 面试高频考点

**面试官常问（综合实战类）**：
- "给你一组实验数据，你会怎么做分析？"
- "实验组转化率显著提升了，但留存率下降了，你怎么看？"
- "如果实验结果是'既显著又无业务价值'，你怎么建议？"

## 🧩 区块2：知识点详细拆解——每个分析步骤的技术细节

### 2.1 实验1设计：新用户引导流程

**业务假设**：
- 我们相信：在注册完成后增加3步兴趣引导
- 会导致：新用户7日留存率
- 发生变化：从25%提升到27.5%（+10%相对提升）
- 证据：用户调研显示"注册后不知道做什么"是流失第一原因

**实验设计**：
- 随机化单元：新注册用户（user_id）
- 分流比例：50:50
- 主要指标：7日留存率（D7 Retention）
- 次要指标：D1留存、D3留存、首日活跃时长
- 护栏指标：注册完成率（不能因引导步骤而下降）、卸载率

### 2.2 实验2设计：推荐算法迭代

**业务假设**：
- 我们相信：基于Transformer的新推荐模型
- 会导致：人均GMV
- 发生变化：从200元提升到210元（+5%相对提升）
- 证据：离线Recall@100提升12%

**实验设计**：
- 随机化单元：活跃用户（user_id）
- 分流比例：50:50
- 主要指标：人均GMV（连续指标）
- 次要指标：点击率、转化率、推荐多样性
- 护栏指标：页面加载时间P99、崩溃率

### 2.3 连续指标的分析方法

**为什么重要**：GMV、时长、订单数等连续指标的分析与比例指标不同。

| 维度 | 比例指标 | 连续指标 |
|------|---------|----------|
| 数据类型 | 0/1二值 | 连续值 |
| 检验方法 | t检验/z检验（比例） | t检验 |
| 效应量 | 绝对差值/相对提升 | Cohen's d |
| 方差估计 | p(1-p) | 直接计算方差 |
| 常见问题 | 基线p接近0或1时方差小 | 偏态分布（离群值影响大） |

**GMV类指标的特殊处理**：
- 高度偏态（少数大额订单拉高均值）
- 常用处理：截断（Winsorization）→ 取对数（log transform）→ 或用中位数替代均值
- CUPED：用实验前的人均GMV做协变量

### 2.4 离群值处理策略

**为什么重要**：不处理离群值可能导致错误结论。

| 方法 | 做法 | 优点 | 缺点 |
|------|------|------|------|
| 截断(Winsorization) | 将>P99的值替换为P99 | 保留数据不丢失 | 改变了分布尾部 |
| 剔除(Trimming) | 去掉>P99的数据 | 简单 | 丢失数据 |
| 对数变换 | y' = log(y+1) | 压缩尺度 | 解释不直观 |
| 分层处理 | 按金额分层分别分析 | 保留信息 | 复杂度高 |

**AB实验中的最佳实践**：
1. 事前定义离群值规则（在实验设计文档中）
2. 主要分析用截断数据，敏感性分析用原始数据
3. 两组用同样的离群值规则
4. 报告离群值的比例（如果两组离群值比例显著不同，这本身就是一个信号）

### 2.5 异质性分析——谁受益最大？

**为什么重要**：整体不显著不代表所有子群都不显著；整体显著也不代表所有子群都受益。

**常见分层维度**：
- 平台（iOS vs Android）
- 用户新老（新用户 vs 老用户）
- 活跃度（高活 vs 中活 vs 低活）
- 地域（一线城市 vs 非一线）
- 设备（高端机 vs 低端机）

**注意事项**：
- 子群分析标注为"探索性"——不能当作确证性结论
- 子群越多，多重比较问题越严重
- 如果发现某个子群效果特别好/差——提出新假设，设计新实验验证

### 2.6 实验结论的沟通框架

**为什么重要**：再好的分析，沟通不好也等于零。

**实验报告的标准结构**：
```
1. 一句话结论（TL;DR）
2. 实验背景和目标
3. 实验设计概要
4. 数据质量（SRM、缺失率、离群值）
5. 主要结果（p值+效应量+95%CI）
6. 护栏指标结果
7. 探索性分析（子群分析等）
8. 建议和下一步
```

**TL;DR模板**：
- "新引导流程使7日留存率从25.0%提升到27.3%（+2.3pp, 95%CI [1.5, 3.1]pp, p<0.001），护栏指标均未恶化。建议全量上线。"
- "新推荐算法对人均GMV的提升不显著（+3.2元, 95%CI [-5.1, 11.5], p=0.45），且页面加载时间P99显著恶化。建议不上线。"

## 💻 区块3：动手练习——实验1：比例指标（新用户引导）

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print("="*60)
print("实验1：新用户注册引导流程AB实验")
print("="*60)

# ===== 步骤1：计算样本量 =====
print("\n步骤1：计算样本量")
baseline_ret = 0.25  # 基线7留25%
mde_rel = 0.10       # 10%相对提升
alpha = 0.05
power = 0.80

def sample_n(p, mde_rel, a=0.05, pw=0.80):
    za = stats.norm.ppf(1 - a/2)
    zb = stats.norm.ppf(pw)
    p1 = p
    p2 = p * (1 + mde_rel)
    pb = (p1 + p2) / 2
    n = (za * np.sqrt(2*pb*(1-pb)) + zb * np.sqrt(p1*(1-p1)+p2*(1-p2)))**2 / (p2 - p1)**2
    return int(np.ceil(n))

n_per_group = sample_n(baseline_ret, mde_rel)
print(f"  基线7留: {baseline_ret:.0%}")
print(f"  MDE: +{mde_rel:.0%}相对 = +{baseline_ret*mde_rel:.1%}绝对")
print(f"  每组需要: {n_per_group:,} 新用户")
print(f"  两组共需: {n_per_group*2:,} 新用户")
print(f"  日新用户约3000 → 需要{n_per_group*2/3000:.0f}天积累")

# ===== 步骤2：模拟实验数据 =====
print("\n步骤2：模拟实验数据（假设真实提升2.5pp）")
true_lift = 0.025  # 真实绝对提升2.5pp
n = n_per_group

control_ret = np.random.binomial(1, baseline_ret, n)
experiment_ret = np.random.binomial(1, baseline_ret + true_lift, n)

# ===== 步骤3：SRM检验 =====
print("\n步骤3：SRM检验")
total = len(control_ret) + len(experiment_ret)
expected = total / 2
chi2_srm = ((len(control_ret) - expected)**2 / expected +
            (len(experiment_ret) - expected)**2 / expected)
p_srm = 1 - stats.chi2.cdf(chi2_srm, 1)
print(f"  对照组={len(control_ret):,}, 实验组={len(experiment_ret):,}")
print(f"  p(SRM)={p_srm:.4f} → {'✗ SRM！' if p_srm < 0.05 else '✓ 通过'}")

# ===== 步骤4：主要指标分析 =====
print("\n步骤4：主要指标分析（7日留存率）")
t_stat_ret, p_ret = stats.ttest_ind(experiment_ret, control_ret)
effect_ret = experiment_ret.mean() - control_ret.mean()
effect_rel = experiment_ret.mean() / control_ret.mean() - 1
se_ret = np.sqrt(control_ret.var()/n + experiment_ret.var()/n)
ci_ret = stats.t.interval(0.95, 2*n-2, loc=effect_ret, scale=se_ret)

print(f"  对照组7留: {control_ret.mean():.2%}")
print(f"  实验组7留: {experiment_ret.mean():.2%}")
print(f"  绝对提升: {effect_ret*100:.2f}pp")
print(f"  相对提升: {effect_rel*100:.1f}%")
print(f"  95% CI: [{ci_ret[0]*100:.2f}pp, {ci_ret[1]*100:.2f}pp]")
print(f"  p值: {p_ret:.4f} → {'显著' if p_ret < 0.05 else '不显著'}")

# ===== 步骤5：护栏指标检查 =====
print("\n步骤5：护栏指标检查")

# 护栏1：注册完成率（不希望下降）
control_signup = np.random.binomial(1, 0.80, n)
exp_signup = np.random.binomial(1, 0.80, n)  # H₀为真：引导不影响注册完成率
_, p_signup = stats.ttest_ind(exp_signup, control_signup)
print(f"  注册完成率: 对照={control_signup.mean():.1%}, 实验={exp_signup.mean():.1%}")
print(f"    p={p_signup:.4f} → {'⚠ 护栏恶化！' if p_signup < 0.05 else '✓ 通过'}")

# 护栏2：卸载率（不希望上升）
control_uninstall = np.random.binomial(1, 0.05, n)
exp_uninstall = np.random.binomial(1, 0.05, n)  # H₀为真
_, p_uninstall = stats.ttest_ind(exp_uninstall, control_uninstall)
print(f"  卸载率: 对照={control_uninstall.mean():.1%}, 实验={exp_uninstall.mean():.1%}")
print(f"    p={p_uninstall:.4f} → {'⚠ 护栏恶化！' if p_uninstall < 0.05 else '✓ 通过'}")

# ===== 步骤6：决策 =====
print("\n步骤6：决策矩阵")
print("="*40)

practical_threshold = 0.01  # 1pp的业务意义阈值
srm_ok = p_srm >= 0.05
guardrail_ok = p_signup >= 0.05 and p_uninstall >= 0.05
significant = p_ret < 0.05
practically_meaningful = effect_ret > practical_threshold

if not srm_ok:
    print("决策: ✗ 实验无效（SRM不通过），修bug后重新实验")
elif not guardrail_ok:
    print("决策: ✗ 不上线（护栏指标恶化——一票否决）")
elif significant and practically_meaningful:
    print(f"决策: ✓ 建议全量上线新版引导流程")
    print(f"  预期7留提升: {ci_ret[0]*100:.1f}pp ~ {ci_ret[1]*100:.1f}pp")
    print(f"  后续监控: 全量上线后继续监控7天")
elif significant and not practically_meaningful:
    print(f"决策: △ 虽显著但效应太小(<{practical_threshold:.0%})")
    print(f"  不建议上线——ROI不足以覆盖工程成本")
elif not significant and practically_meaningful:
    print(f"决策: △ 效应有业务价值但不显著")
    print(f"  建议: 延长实验2周或增加流量")
else:
    print(f"决策: ✗ 不上线（不显著且效应小）")
    print(f"  建议: 从用户调研中寻找新的优化方向")

### 实验2：连续指标（推荐算法GMV）

In [ ]:
print("\n" + "="*60)
print("实验2：推荐算法迭代AB实验（连续指标）")
print("="*60)

np.random.seed(123)

# ===== 样本量计算（连续指标） =====
print("\n步骤1：计算样本量（连续指标）")
baseline_gmv = 200  # 人均GMV
gmv_std = 500       # GMV标准差
mde_abs = 10        # 期望检测到10元的提升

za = stats.norm.ppf(1 - 0.05/2)
zb = stats.norm.ppf(0.80)
n_continuous = int(np.ceil(2 * (za + zb)**2 * gmv_std**2 / mde_abs**2))
print(f"  基线人均GMV: {baseline_gmv}元, 标准差: {gmv_std}元")
print(f"  MDE: {mde_abs}元 (+{mde_abs/baseline_gmv:.0%})")
print(f"  每组需要: {n_continuous:,} 用户")

# ===== 模拟GMV数据（偏态分布） =====
print("\n步骤2：模拟偏态GMV数据")
n_cont = n_continuous

# 用对数正态模拟GMV的偏态分布
sigma_sq = np.log(1 + gmv_std**2 / baseline_gmv**2)
mu = np.log(baseline_gmv) - sigma_sq / 2

control_gmv = np.random.lognormal(mu, np.sqrt(sigma_sq), n_cont)
experiment_gmv = np.random.lognormal(mu, np.sqrt(sigma_sq), n_cont)
experiment_gmv *= 1.03  # 真实+3%提升

print(f"  对照组: 均值={control_gmv.mean():.1f}, 中位数={np.median(control_gmv):.1f}")
print(f"  实验组: 均值={experiment_gmv.mean():.1f}, 中位数={np.median(experiment_gmv):.1f}")
print(f"  对照组偏度: {stats.skew(control_gmv):.2f} (0=正态, >0=右偏)")

# ===== 分析：原始数据 =====
print("\n步骤3：分析——原始数据")
t_gmv, p_gmv = stats.ttest_ind(experiment_gmv, control_gmv)
effect_gmv = experiment_gmv.mean() - control_gmv.mean()
se_gmv = np.sqrt(control_gmv.var()/n_cont + experiment_gmv.var()/n_cont)
ci_gmv = stats.t.interval(0.95, 2*n_cont-2, loc=effect_gmv, scale=se_gmv)
cohens_d = effect_gmv / np.sqrt((control_gmv.var() + experiment_gmv.var())/2)

print(f"  人均GMV差值: {effect_gmv:.2f}元")
print(f"  相对提升: {effect_gmv/baseline_gmv*100:.1f}%")
print(f"  95% CI: [{ci_gmv[0]:.1f}, {ci_gmv[1]:.1f}]元")
print(f"  Cohen's d: {cohens_d:.4f}")
print(f"  p值: {p_gmv:.4f} → {'显著' if p_gmv < 0.05 else '不显著'}")

# ===== 分析：截断处理 =====
print("\n步骤4：分析——截断处理（去除P99以上）")
cap = np.percentile(np.concatenate([control_gmv, experiment_gmv]), 99)
control_winsor = np.clip(control_gmv, 0, cap)
exp_winsor = np.clip(experiment_gmv, 0, cap)

t_w, p_w = stats.ttest_ind(exp_winsor, control_winsor)
effect_w = exp_winsor.mean() - control_winsor.mean()
print(f"  截断后人均GMV差值: {effect_w:.2f}元 (原始: {effect_gmv:.2f}元)")
print(f"  p值: {p_w:.4f} → {'显著' if p_w < 0.05 else '不显著'}")
print(f"  截断阈值(P99): {cap:.0f}元")
print(f"  → 截断对结论的影响取决于离群值的分布是否均衡")

# ===== 简化版CUPED =====
print("\n步骤5：简化版CUPED（用'实验前GMV'做协变量）")
# 模拟实验前GMV（与实验期GMV相关系数0.6）
correlation = 0.6
control_pre = baseline_gmv + correlation * (control_gmv - baseline_gmv) + np.random.normal(0, gmv_std*np.sqrt(1-correlation**2), n_cont)
exp_pre = baseline_gmv + correlation * (experiment_gmv - baseline_gmv) + np.random.normal(0, gmv_std*np.sqrt(1-correlation**2), n_cont)

# CUPED调整
theta = np.cov(control_gmv, control_pre)[0, 1] / np.var(control_pre)
control_cuped = control_gmv - theta * (control_pre - control_pre.mean())
exp_cuped = experiment_gmv - theta * (exp_pre - exp_pre.mean())

print(f"  实验前-实验期GMV相关系数: {np.corrcoef(control_gmv, control_pre)[0,1]:.3f}")
print(f"  原始方差: 对照={control_gmv.var():.0f}, 实验={experiment_gmv.var():.0f}")
print(f"  CUPED后方差: 对照={control_cuped.var():.0f}, 实验={exp_cuped.var():.0f}")
print(f"  方差缩减: {(1-control_cuped.var()/control_gmv.var())*100:.0f}%")

t_cuped, p_cuped = stats.ttest_ind(exp_cuped, control_cuped)
print(f"  CUPED后p值: {p_cuped:.4f} (原始p值: {p_gmv:.4f})")
print(f"  → CUPED通过减少方差，使检验更敏感")

### 步骤3：异质性分析

In [ ]:
print("\n" + "="*60)
print("异质性分析：实验1按平台分层")
print("="*60)

np.random.seed(456)

# 模拟：iOS用户对新引导更敏感
n_ios = n // 3
n_android = n - n_ios

# iOS：大效果
c_ios = np.random.binomial(1, baseline_ret, n_ios)
e_ios = np.random.binomial(1, baseline_ret + 0.04, n_ios)  # +4pp

# Android：小效果
c_android = np.random.binomial(1, baseline_ret, n_android)
e_android = np.random.binomial(1, baseline_ret + 0.01, n_android)  # +1pp

print(f"{'子群':<12} {'对照组':<10} {'实验组':<10} {'差值':<12} {'p值':<10} {'样本量':<10}")
print("-" * 64)

for label, c, e, size in [("iOS", c_ios, e_ios, n_ios),
                           ("Android", c_android, e_android, n_android),
                           ("总体", control_ret, experiment_ret, n)]:
    _, p = stats.ttest_ind(e, c)
    diff = e.mean() - c.mean()
    print(f"{label:<12} {c.mean():<10.2%} {e.mean():<10.2%} {diff*100:<12.2f}pp {p:<10.4f} {size:<10,}")

# 交互效应检验
print(f"\n交互效应分析:")
print(f"  iOS效果: {(e_ios.mean() - c_ios.mean())*100:.1f}pp")
print(f"  Android效果: {(e_android.mean() - c_android.mean())*100:.1f}pp")
print(f"  差异之差异: {(e_ios.mean() - c_ios.mean() - (e_android.mean() - c_android.mean()))*100:.1f}pp")
print(f"\n→ iOS效果似乎更大，但这是探索性分析")
print(f"→ 需要设计新实验来验证'iOS用户对引导更敏感'这个假设")
print(f"→ 不能在当前实验中基于子群结果做差异化上线决策")

In [ ]:
print("\n" + "="*60)
print("最终：生成标准化实验报告")
print("="*60)

def generate_report(exp_name, control, experiment, guardrail_results, 
                    practical_threshold, alpha=0.05):
    """生成标准化实验报告"""
    n = len(control)
    t, p = stats.ttest_ind(experiment, control)
    effect = experiment.mean() - control.mean()
    se = np.sqrt(control.var()/n + experiment.var()/n)
    ci = stats.t.interval(0.95, 2*n-2, loc=effect, scale=se)

    report = []
    report.append(f"{'='*50}")
    report.append(f"实验报告: {exp_name}")
    report.append(f"{'='*50}")
    report.append(f"\n【TL;DR】")

    if p < alpha and effect > practical_threshold:
        report.append(f"✓ 建议上线。")
    elif p < alpha and effect <= practical_threshold:
        report.append(f"△ 虽显著但效应太小，不建议上线。")
    elif p >= alpha and effect > practical_threshold:
        report.append(f"△ 效应有业务价值但不显著，建议延长实验。")
    else:
        report.append(f"✗ 不显著且效应小，不建议上线。")

    report.append(f"\n【主要结果】")
    report.append(f"  对照组: {control.mean():.3%}")
    report.append(f"  实验组: {experiment.mean():.3%}")
    report.append(f"  绝对差值: {effect*100:.2f}pp")
    report.append(f"  95% CI: [{ci[0]*100:.2f}pp, {ci[1]*100:.2f}pp]")
    report.append(f"  p值: {p:.4f} {'(显著)' if p < alpha else '(不显著)'}")
    report.append(f"  每组样本量: {n:,}")

    report.append(f"\n【护栏指标】")
    for name, passed in guardrail_results.items():
        report.append(f"  {name}: {'✓ 通过' if passed else '✗ 恶化！'}")

    return "\n".join(report)

# 生成报告
guardrails = {
    "注册完成率": p_signup >= 0.05,
    "卸载率": p_uninstall >= 0.05,
    "SRM": p_srm >= 0.05
}

report = generate_report(
    "新用户注册引导流程优化",
    control_ret, experiment_ret,
    guardrails, practical_threshold=0.01
)
print(report)

## 📊 区块4：业务场景应用

### 场景1：实验组转化率涨了但留存跌了——怎么办？

**问题**：某电商测试"首单立减20元"策略。转化率涨了3pp（p<0.001），但7日留存跌了5pp（p<0.01）。PM说"转化率涨了就是好的，上线吧"。

**分析**：这是典型的"指标权衡（Metric Trade-off）"场景。
- 首单立减吸引来的用户可能只是"薅羊毛"——拿完优惠就走
- 短期转化涨了，但用户质量降低了（长期价值为负）
- 如果留存是护栏指标，这个实验应该被护栏指标一票否决

**决策框架**：
1. 如果留存是护栏指标 → 一票否决，不上线
2. 如果留存是次要指标 → 需要计算"长期价值 = 首单价值 + 留存概率 × 后续价值"，做一个综合ROI判断
3. 无论如何，转化率和留存方向相反是一个重要信号——这个策略可能在牺牲长期健康换取短期指标

**决策**：不上线。重新设计策略——比如"满3单返20元"，既给优惠又鼓励复购。

### 场景2：实验不显著但PM坚持要上线

**问题**：实验跑完，主要指标p=0.12。PM说"效果方向是对的（+2pp），只是样本不够，我们上线后继续观察"。

**分析**：
- p=0.12意味着如果策略无效，有12%的概率看到这样的数据
- "方向是对的"是一种危险的直觉——很多策略的"方向正确"只是随机波动
- "上线后继续观察"的问题：失去了对照组，再也无法判断效果

**应对策略**：
1. 如果PM真的相信效果存在，应该同意延长实验：加大样本量（或延长实验时间），用更充分的证据来做决策
2. 如果资源不够延长实验，坦诚地说："以目前的证据强度，我们无法判断这个策略是否有效。上线后也无法判断（因为没有对照组了）"
3. 把球踢回给PM："如果策略无效，上线后指标的提升可能是自然波动或其他因素导致的。你愿意承担'把无效策略当成有效'的风险吗？"

**决策**：延长实验2周，积累更多样本后再决策。

### 场景3：多个指标的信号互相矛盾

**问题**：某策略的主要指标（下单率）显著提升(p=0.01)，次要指标（客单价）显著下降(p=0.02)，护栏指标全部通过。怎么办？

**分析**：
- 下单率↑ + 客单价↓ = 可能用户买了更多但更便宜的东西
- 需要计算综合指标：人均GMV = 下单率 × 客单价
- 如果人均GMV方向不明确，可以考虑在实验设计阶段就定义为主要指标

**决策**：
1. 计算人均GMV的综合效应
2. 分析客单价下降的原因——是品类结构变了还是折扣率变了？
3. 如果综合效应为正且有业务价值，可以上线
4. 如果综合效应不明确，建议不上线

### 场景4：实验"完美成功"——为什么反而要警惕？

**问题**：某实验所有指标都显著向好：转化率+5pp(p<0.001)、留存+3pp(p<0.001)、GMV+8%(p<0.001)、护栏全部通过。完美的实验结果！

**分析**：在实验领域，"太好的结果往往不是真的"。需要考虑：
1. 新奇效应（Novelty Effect）：用户因为变化本身而反应，不是策略真的更好
2. 选择偏误：实验组是否有异常的用户筛选？（SRM已通过？）
3. 同期事件：实验期间是否有外部事件（节日、竞对事件）恰好影响了实验组？
4. 数据bug：指标计算是否有误？

**应对**：
1. 延长实验时间——新奇效应会随时间衰减
2. 检查数据采集管道是否有bug
3. 按时间分段分析——效果是否随时间衰减？
4. 上线后继续监控——如果效果真的这么大，上线后应该持续保持

**决策**：延长实验1-2周验证效果持续性。如果效果持续存在且稳定，再全量上线。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能独立完成一个比例指标的AB实验分析（从数据到结论）
- [ ] 能计算并报告p值、效应量、置信区间
- [ ] 能进行SRM检验
- [ ] 能写出TL;DR一句话结论

**答案提示**：
- 完整流程：SRM→主要指标→效应量+CI→护栏→决策
- 三件套：p值+效应量+95%CI
- SRM用卡方检验
- TL;DR格式："X策略使Y指标从A变为B（+C, 95%CI [D,E], p=F）"

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能处理连续指标的AB实验分析
- [ ] 能合理处理离群值并报告敏感性
- [ ] 能进行子群分析并正确标注为探索性
- [ ] 能撰写标准化的实验报告

**答案提示**：
- 连续指标用t检验，效应量用Cohen's d
- 离群值：事前定义规则+主分析截断+敏感性用原始数据
- 子群分析=探索性分析，不能作为确证性结论
- 实验报告8要素：TL;DR→背景→设计→数据质量→结果→护栏→子群→建议

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能处理指标之间方向矛盾的情况（转化↑留存↓）
- [ ] 能识别"过于完美"的实验结果并排查原因
- [ ] 能对"不显著但有方向"的结果给出正确的建议
- [ ] 能在组织内推动实验结论的规范化沟通

**答案提示**：
- 指标矛盾时看综合ROI和护栏指标
- 完美结果警惕：新奇效应、选择偏误、同期事件、数据bug
- 不显著但有方向：不是"成功了"，是"证据不足"→延长实验
- 规范化：模板化报告+实验数据库+定期复盘

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W7D1-D5全部 | 综合实战整合了本周所有知识 |
| **← 前置依赖** | W1 SQL | 实验数据需要用SQL从数据库中提取 |
| **→ 后续用途** | W8 CUPED进阶 | 系统化的方差缩减技术 |
| **→ 后续用途** | W8 网络效应 | 社交产品和双边市场实验 |
| **→ 后续用途** | W9 长期效应 | 如何评估实验的长期影响 |
| **业务连接** | 所有产品决策 | 这是数据分析师日常工作中最重要的技能之一 |

### 横向连接：与其他分析方法的关联

**与SQL的连接**：
- 实验数据存储在数据库中，需要SQL提取
- 指标计算通常用SQL定义
- 实验分析可以SQL化（写一个标准的实验分析SQL模板）

**与看板/可视化的连接**：
- 实验结果应该归档到实验数据库
- 建立实验dashboard（正在运行的实验、本周结论）
- 实验结论的可视化（效应量森林图）

**与机器学习的连接**：
- ML模型的在线效果评估就是AB实验
- 实验可以同时比较多个模型变体（A/B/n + 贝叶斯方法）
- 异质性处理效应（HTE）可以用ML方法估计

## 📝 区块7：当日小结

> **今天完成了两个端到端的AB实验：比例指标（新用户引导→7留）和连续指标（推荐算法→GMV）。完整流程包括：样本量计算→SRM检验→主要指标分析（p值+效应量+CI）→护栏指标检查→异质性分析→决策。关键认知：实验结论是"多维度判断"而非"一个p值判断"。SRM不过=无效，护栏恶化=一票否决，显著但效应小=不上线。离群值处理对连续指标至关重要。子群分析永远是探索性的，不能作为确证性结论。**

**三条金句带回家**：
1. 一个完整的AB实验结论 = SRM ✓ + 主要指标(效应量+CI+p) + 护栏 ✓ + 业务判断
2. 太好的实验结果往往不是真的——新奇效应、选择偏误、数据bug，总有一个在搞鬼
3. 子群分析是"假设生成器"，不是"决策依据"——发现子群差异 → 提出新假设 → 设计新实验

### 🔖 课后拓展

1. **练习**：用真实数据（或更复杂的模拟数据）完成一个AB实验的全流程分析。包含：比例指标+连续指标、SRM检测、护栏检查、异质性分析、实验报告撰写。

2. **思考**：如果实验的主要指标是"用户生命周期价值(LTV)"，但LTV需要6个月的计算窗口，你怎么设计实验？

3. **预习**：明天周复习——回顾本周AB实验基础的全部知识点。

4. **面试准备**：背诵以下面试题答案：
   - "给你一组实验数据，你会怎么做分析？"（7步分析框架）
   - "实验组转化率涨了但留存跌了，你怎么决策？"（护栏一票否决+ROI综合判断）
   - "实验不显著但PM想上线，你怎么处理？"（延长实验/坦诚沟通风险）

---

*03analyst_thinking · Week 7 Day 6 · 2026-05-21*